# Variables normalizadas

Este notebook carga `firms_spain_final.csv` y normaliza las variables numéricas continuas con **Z-score** (`StandardScaler`): centra cada variable alrededor de 0 y la escala para que su desviación típica sea aproximadamente 1.

Se excluyen identificadores y códigos numéricos categóricos como `type`, `sensor`, `acq_time` y las columnas `clc_`; normalizar esos códigos les daría un significado de magnitud que no tienen. Las variables categóricas y los valores faltantes se conservan. El CSV de salida es una copia donde solo se sustituyen las columnas seleccionadas.

La celda de exportación ajusta el escalador al conjunto completo porque genera una versión normalizada descriptiva del archivo. Si se usa para evaluar un modelo predictivo, hay que dividir primero entrenamiento/prueba y ajustar el escalador solo con entrenamiento, como muestra la última sección.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from IPython.display import display

csv_path = Path.home() / 'Desktop' / 'wildfires-project-development' / 'data' / 'processed' / 'enriched' / 'firms_spain_final.csv'
if not csv_path.exists():
    raise FileNotFoundError(f'No se encontró el CSV: {csv_path}')
df = pd.read_csv(csv_path, low_memory=False)
print(f'{len(df):,} filas y {df.shape[1]} columnas')
df.head()

## Seleccionar variables

Además de las columnas CLC, se excluyen columnas que son códigos o etiquetas, aunque estén almacenadas como números. Las variables booleanas también quedan fuera. La lista de exclusión se puede ajustar según el uso posterior.

In [ ]:
exclude = {'type', 'sensor', 'acq_time', 'is_day'}
exclude.update(c for c in df.columns if c.lower().startswith('clc_'))
numeric_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in exclude and not pd.api.types.is_bool_dtype(df[c])
]
if not numeric_cols:
    raise ValueError('No se encontraron variables numéricas continuas para normalizar.')
print(f'Variables numéricas seleccionadas ({len(numeric_cols)}):')
print(numeric_cols)
display(pd.DataFrame({
    'faltantes': df[numeric_cols].isna().sum(),
    'valores_distintos': df[numeric_cols].nunique(dropna=True),
    'media_original': df[numeric_cols].mean(),
    'desviacion_original': df[numeric_cols].std()
}).sort_index())

## Aplicar Z-score y guardar copia

`StandardScaler` conserva los NaN. No imputa faltantes: si el siguiente modelo los requiere, habrá que imputarlos dentro del pipeline de entrenamiento.

In [ ]:
scaler = StandardScaler()
df_norm = df.copy()
df_norm[numeric_cols] = scaler.fit_transform(df[numeric_cols])

comparison = pd.DataFrame({
    'media_normalizada': df_norm[numeric_cols].mean(),
    'desviacion_normalizada': df_norm[numeric_cols].std(),
    'faltantes': df_norm[numeric_cols].isna().sum()
})
display(comparison)

output_path = csv_path.with_name('firms_spain_final_normalized.csv')
df_norm.to_csv(output_path, index=False)
print(f'Archivo normalizado guardado en: {output_path}')
df_norm.head()

## Ejemplo correcto para evaluación predictiva

Al evaluar un modelo, ajustar el escalador con todo el CSV filtraría información del conjunto de prueba. En este ejemplo se divide primero y se aprende la media/desviación únicamente en entrenamiento; después se transforma prueba con esos mismos parámetros. Para una red neuronal, lo ideal es integrar este escalador en un `Pipeline`.

In [ ]:
X = df[numeric_cols]
X_train, X_test = train_test_split(X, test_size=0.20, random_state=42)
scaler_train = StandardScaler()
X_train_norm = pd.DataFrame(scaler_train.fit_transform(X_train), columns=numeric_cols, index=X_train.index)
X_test_norm = pd.DataFrame(scaler_train.transform(X_test), columns=numeric_cols, index=X_test.index)
print('Filas entrenamiento:', len(X_train_norm), '| filas prueba:', len(X_test_norm))
display(X_train_norm.head())